# Compression Experiment

Goal: Evaluate inverted-index compression on a larger document collection.

We will compare:
- Raw 32-bit docIDs
- d-gap + Variable Byte encoding
- Compression ratio
- Encoding time
- Decoding time

**Status:** Implemented compression experiment. Uses the first 1,000 training documents from the publicly available 20 Newsgroups dataset via scikit-learn. Results depend on the dataset version and execution environment.

In [1]:
from sklearn.datasets import fetch_20newsgroups


In [2]:
newsgroups = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

In [ ]:
print(len(newsgroups.data))
print(newsgroups.data[0][:500])

In [5]:
sample_documents = {}
for i in range(1000):
    sample_documents[i + 1] = newsgroups.data[i]
len(sample_documents)

1000

In [7]:
import re
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return tokens

In [8]:
tokenized_documents = {}
for doc_id, text in sample_documents.items():
    tokenized_documents[doc_id] = tokenize(text)
len(tokenized_documents)

1000

In [9]:
inverted_index = {}
for doc_id, tokens in tokenized_documents.items():
    for term in set(tokens):
        if term not in inverted_index:
            inverted_index[term] = [doc_id]
        else:
            inverted_index[term].append(doc_id)

print(len(inverted_index))
print(inverted_index["car"][:20])

19216
[1, 18, 31, 49, 57, 65, 73, 78, 85, 263, 275, 492, 527, 558, 565, 572, 593, 597, 599, 622]


In [10]:
def d_gap_encode(postings):
    if len(postings) == 0:
        return []
    gaps = [postings[0]]
    for i in range(1, len(postings)):
        gaps.append(postings[i] - postings[i - 1])
    return gaps

def d_gap_decode(gaps):
    if len(gaps) == 0:
        return []
    postings = [gaps[0]]
    for i in range(1, len(gaps)):
        postings.append(postings[-1] + gaps[i])
    return postings

def vb_encode_number(n):
    bytes_list = []
    while True:
        bytes_list.insert(0, n % 128)
        if n < 128:
            break
        n = n // 128
    bytes_list[-1] += 128
    return bytes_list

def vb_encode(numbers):
    encoded = []
    for n in numbers:
        encoded.extend(vb_encode_number(n))
    return encoded

In [11]:
car_postings = inverted_index["car"]
car_gaps = d_gap_encode(car_postings)
car_encoded = vb_encode(car_gaps)

print(car_postings[:20])
print(car_gaps[:20])
print(car_encoded[:30])

[1, 18, 31, 49, 57, 65, 73, 78, 85, 263, 275, 492, 527, 558, 565, 572, 593, 597, 599, 622]
[1, 17, 13, 18, 8, 8, 8, 5, 7, 178, 12, 217, 35, 31, 7, 7, 21, 4, 2, 23]
[129, 145, 141, 146, 136, 136, 136, 133, 135, 1, 178, 140, 1, 217, 163, 159, 135, 135, 149, 132, 130, 151, 138, 152, 132, 170, 157, 161, 136, 181]


In [14]:
raw_bytes = 0
compressed_bytes = 0
for term, postings in inverted_index.items():
    raw_bytes += len(postings) * 4
    gaps = d_gap_encode(postings)
    encoded = vb_encode(gaps)
    compressed_bytes += len(encoded)
compression_ratio = compressed_bytes / raw_bytes
space_saving = 1 - compression_ratio
print("Raw bytes:", raw_bytes)
print("Compressed bytes:", compressed_bytes)
print("Compression ratio:", compression_ratio)
print("Space saving:", space_saving)

Raw bytes: 377332
Compressed bytes: 118851
Compression ratio: 0.31497726140375054
Space saving: 0.6850227385962495


In [13]:
import time
start_time = time.perf_counter()
compressed_postings = {}
for term, postings in inverted_index.items():
    gaps = d_gap_encode(postings)
    encoded = vb_encode(gaps)
    compressed_postings[term] = encoded
end_time = time.perf_counter()
encoding_time = end_time - start_time
print("Encoding time:", encoding_time, "seconds")

Encoding time: 0.1032061000005342 seconds


In [15]:
def vb_decode(encoded):
    numbers = []
    n = 0
    for byte in encoded:
        if byte < 128:
            n = 128 * n + byte
        else:
            n = 128 * n + (byte - 128)
            numbers.append(n)
            n = 0
    return numbers

In [16]:
start_time = time.perf_counter()
decoded_postings = {}
for term, encoded in compressed_postings.items():
    gaps = vb_decode(encoded)
    postings = d_gap_decode(gaps)
    decoded_postings[term] = postings
end_time = time.perf_counter()
decoding_time = end_time - start_time
print("Decoding time:", decoding_time, "seconds")

Decoding time: 0.1281530000269413 seconds


In [17]:
decoded_postings == inverted_index

True

In [18]:
import statistics
encoding_times = []
decoding_times = []
for _ in range(20):
    start_time = time.perf_counter()
    compressed_postings = {}
    for term, postings in inverted_index.items():
        gaps = d_gap_encode(postings)
        encoded = vb_encode(gaps)
        compressed_postings[term] = encoded
    end_time = time.perf_counter()
    encoding_times.append(end_time - start_time)
for _ in range(20):
    start_time = time.perf_counter()
    decoded_postings = {}
    for term, encoded in compressed_postings.items():
        gaps = vb_decode(encoded)
        postings = d_gap_decode(gaps)
        decoded_postings[term] = postings
    end_time = time.perf_counter()
    decoding_times.append(end_time - start_time)

In [19]:
print("Encoding mean:", statistics.mean(encoding_times))
print("Encoding std:", statistics.stdev(encoding_times))
print("Decoding mean:", statistics.mean(decoding_times))
print("Decoding std:", statistics.stdev(decoding_times))

Encoding mean: 0.17653486999333837
Encoding std: 0.13403997611138446
Decoding mean: 0.09908631500438787
Decoding std: 0.05025637047790511


In [20]:
# Warm-up
for _ in range(5):
    compressed_postings = {}
    for term, postings in inverted_index.items():
        gaps = d_gap_encode(postings)
        encoded = vb_encode(gaps)
        compressed_postings[term] = encoded

In [21]:
encoding_times = []
for _ in range(30):
    start_time = time.perf_counter()
    compressed_postings = {}
    for term, postings in inverted_index.items():
        gaps = d_gap_encode(postings)
        encoded = vb_encode(gaps)
        compressed_postings[term] = encoded
    end_time = time.perf_counter()
    encoding_times.append(end_time - start_time)

In [22]:
print("Mean:", statistics.mean(encoding_times))
print("Median:", statistics.median(encoding_times))
print("Std:", statistics.stdev(encoding_times))

Mean: 0.14695828665862792
Median: 0.12386299998615868
Std: 0.06293865513533306


In [23]:
for _ in range(5):
    decoded_postings = {}
    for term, encoded in compressed_postings.items():
        gaps = vb_decode(encoded)
        postings = d_gap_decode(gaps)
        decoded_postings[term] = postings

In [24]:
decoding_times = []
for _ in range(30):
    start_time = time.perf_counter()
    decoded_postings = {}
    for term, encoded in compressed_postings.items():
        gaps = vb_decode(encoded)
        postings = d_gap_decode(gaps)
        decoded_postings[term] = postings
    end_time = time.perf_counter()
    decoding_times.append(end_time - start_time)

In [25]:
print("Mean:", statistics.mean(decoding_times))
print("Median:", statistics.median(decoding_times))
print("Std:", statistics.stdev(decoding_times))

Mean: 0.12161305334108571
Median: 0.09038270002929494
Std: 0.06869652402280353
